In [4]:
from langchain.vectorstores import Chroma
from langchain_community.embeddings import ModelScopeEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.memory import ChatMessageHistory
from langchain.prompts.chat import ChatPromptTemplate,SystemMessagePromptTemplate,HumanMessagePromptTemplate,AIMessagePromptTemplate,MessagesPlaceholder
from langchain.schema import HumanMessage,SystemMessage,AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
import os
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

In [6]:
# 定义 Embeddings
embeddings = HuggingFaceEmbeddings(model_name="remote_models/embedding_model")

# 向量数据库持久化路径
persist_directory = 'data_base/vector_db/chroma'

# 加载数据库
vectordb = Chroma(
    persist_directory=persist_directory, 
    embedding_function=embeddings
)

In [38]:
question="推荐一下广州有哪些福利好，有关运维的工作。"
sim_docs = vectordb.similarity_search(question,k=3)
print(f"检索到的内容数：{len(sim_docs)}")

检索到的内容数：3


In [ ]:
for i, sim_doc in enumerate(sim_docs):
    print(f"检索到的第{i}个内容: \n{sim_doc.page_content[:200]}", end="\n--------------\n")

In [11]:
from langchain.prompts import PromptTemplate

# template = """基于以下已知信息，简洁和专业的来回答用户的问题。
#             如果无法从中得到答案，请说 "根据已知信息无法回答该问题" 或 "没有提供足够的相关信息"，不允许在答案中添加编造成分。
#             答案请使用中文。
#             总是在回答的最后说“谢谢你的提问！”。
# 已知信息：{context}
# 问题: {question}"""
template = """使用以下上下文来回答最后的问题。如果你不知道答案，就说你不知道，不要试图编造答
案。最多使用三句话。尽量使答案简明扼要。总是在回答的最后说“谢谢你的提问！”。
{context}
问题: {question}
有用的回答:"""

QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context","question"],
                                 template=template)

# 运行 chain

In [10]:
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={'k': 4})  #默认similarity，k=4

In [45]:
import requests
from typing import Any, List, Mapping, Optional
from langchain.callbacks.manager import CallbackManagerForLLMRun
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.llms.base import LLM

callback_handler = StreamingStdOutCallbackHandler()

class CustomLLM(LLM):
    endpoint: str = "http://localhost:8000"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        callbacks: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        headers = {"Content-Type": "application/json"}
        
        # Prepare the data to send to the server
        data = {
            'query': '推荐一下广州有哪些福利好，有关运维的工作。'  # Use the provided prompt instead of a static query
        }

        # Send the POST request with stream=True for streaming response
        response = requests.post(url=self.endpoint + "/chat", json=data, headers=headers, stream=True)
        response.raise_for_status()

        # Collecting the streamed response line by line
        result = []
        for chunk in response.iter_lines():
            if chunk:
                result.append(chunk.decode('utf-8'))  # Decoding the chunk and adding to the result list

        # Join the collected chunks into one text
        text = ''.join(result)
        return text
    
    @property
    def _llm_type(self) -> str:
        return "QwenLM"



In [57]:
from langchain.chains import RetrievalQA
# 自定义 QA 链
qa_chain = RetrievalQA.from_chain_type(llm=CustomLLM(),
                                        retriever=retriever,
                                        chain_type_kwargs={"prompt":QA_CHAIN_PROMPT},
                                        callbacks=[callback_handler])

In [ ]:
response = qa_chain.run(question)
print()

# for line in response.iter_lines():
#         if line:
#             print(f"Received from server (stream=True): {line.decode('utf-8')}")


{'text': '{"text":["广州有很多福利好的运维工作岗位，例如：\\n\\n1. 广州移动：拥有全国领先的云计算、大数据技术，提供完善的员工福利。\\n\\n2. 中国电信广东公司：在云、大数据等领域有深厚的技术积累，为员工提供了全面的福利待遇。\\n\\n3. 广州京东物流科技有限公司：为员工提供了丰富的培训和发展机会，同时也提供了良好的薪酬福利。\\n\\n4. 广州银行信息技术部：有着丰富的企业文化和社会责任感，为员工提供了各种福利和奖励。\\n\\n5. 广东省通信管理局：拥有国内领先的信息网络建设技术和安全体系，为员工提供了多元化的福利。\\n\\n以上只是一些常见的例子，具体的福利待遇还需要根据不同的公司和职位来确定。建议您在寻找工作时多做了解，以便选择最适合您的职位。"]}'}
